# CS Frotas - Pipeline End-to-End no BigQuery Studio / Colab Enterprise

Este Jupyter Notebook executa o pipeline completo do projeto **CS Frotas** diretamente no ambiente do **BigQuery Studio / Dataform / Colab Enterprise**.

### Etapas do Pipeline:
1. **Ingestão e Tratamento**: Leitura dos relatórios do VETOR e SAP, sanitização de esquemas e carga no BigQuery.
2. **Conciliação e Cruzamento SQL**: Criação da View analítica de divergências de estoque e valores.
3. **Auditoria e IA com Gemini (BigQuery ML)**: Análise preditiva de sobrepreço e anomalias com IA Generativa via SQL.

## 1. Configuração do Ambiente e Instalação de Dependências

In [ ]:
# Instalar dependências se executado no Google Colab Enterprise / BigQuery Studio
!pip install --upgrade google-cloud-bigquery pandas openpyxl db-dtypes -q

In [ ]:
import os
import re
import pandas as pd
from google.cloud import bigquery

# Definição de Variáveis do Projeto
PROJECT_ID = "cs-demo-2026"
DATASET_ID = "cs_frotas_data"

client = bigquery.Client(project=PROJECT_ID)
print(f"✅ Conectado ao BigQuery no projeto: {PROJECT_ID}")

## 2. Função de Ingestão e Sanitização de Colunas

In [ ]:
def sanitize_column_name(col, idx):
    """Sanitiza nomes de colunas para garantir compatibilidade com o BigQuery."""
    if not col or str(col).strip() == "" or str(col) == "None":
        return f"coluna_{idx}"
    s = str(col).strip()
    s = s.replace("ª", "").replace("º", "").replace("%", "pct").replace("/", "_").replace(".", "")
    s = re.sub(r"[^\w\s]", "_", s)
    s = re.sub(r"\s+", "_", s).lower()
    if re.match(r"^\d", s):
        s = "c_" + s
    return s

def ingest_to_bigquery(df, table_name):
    """Envia DataFrame limpo para o BigQuery."""
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    print(f"🚀 Enviando {len(df)} registros para {table_ref}...")
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        autodetect=True
    )
    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()
    print(f"✅ Tabela `{table_name}` criada/atualizada no BigQuery!")

## 3. Ingestão dos Relatórios (VETOR e SAP MB52)

In [ ]:
# Caminhos das bases de dados (pode ser ajustado para Google Cloud Storage ex: gs://cs-demo-2026-bucket/...)
file_vetor = "./data/Manutenção - Relatório de Item VETOR.xlsx"
file_sap = "./data/Manutenção - Relatório de Estoque SAP.xlsx"

# 1. Processar VETOR
print("📖 Carregando Relatório VETOR...")
df_vetor = pd.read_excel(file_vetor, sheet_name="Relatorio_de_Item", dtype=str)
df_vetor.columns = [sanitize_column_name(c, i) for i, c in enumerate(df_vetor.columns)]
for col in df_vetor.columns:
    if any(k in col for k in ["valor", "preco", "desconto", "variacao", "reducao"]):
        df_vetor[col] = pd.to_numeric(df_vetor[col].str.replace(",", "."), errors="coerce")
    elif any(k in col for k in ["quantidade", "idade", "km", "dia", "mes", "ano"]):
        df_vetor[col] = pd.to_numeric(df_vetor[col], errors="coerce")

ingest_to_bigquery(df_vetor, "relatorio_item_vetor")

# 2. Processar SAP MB52
print("📖 Carregando Relatório SAP MB52...")
df_sap = pd.read_excel(file_sap, sheet_name="MB52", dtype=str)
df_sap.columns = [sanitize_column_name(c, i) for i, c in enumerate(df_sap.columns)]
for col in df_sap.columns:
    if any(k in col for k in ["val", "utilizacao", "transito"]):
        df_sap[col] = pd.to_numeric(df_sap[col].str.replace(",", "."), errors="coerce")

ingest_to_bigquery(df_sap, "relatorio_estoque_sap_mb52")

## 4. Cruzamento e Conciliação via SQL (`%%bigquery` Magic)

In [ ]:
%%bigquery --project cs-demo-2026
CREATE OR REPLACE VIEW `cs-demo-2026.cs_frotas_data.vw_cruzamento_vetor_sap` AS
SELECT
  v.`código_item_vetor` AS codigo_item_vetor,
  v.`código_item_sap` AS codigo_item_sap_ref,
  s.material AS codigo_material_sap,
  s.`texto_breve_de_material` AS descricao_sap,
  
  -- Valores e Quantidades
  SAFE_CAST(v.quantidade AS NUMERIC) AS qtd_vetor,
  SAFE_CAST(s.`utilização_livre` AS NUMERIC) AS qtd_sap_livre,
  SAFE_CAST(v.valor_total AS NUMERIC) AS valor_total_vetor,
  SAFE_CAST(s.valutilizlivre AS NUMERIC) AS valor_total_sap,
  
  -- Cálculo de Divergência de Valores
  (SAFE_CAST(v.valor_total AS NUMERIC) - SAFE_CAST(s.valutilizlivre AS NUMERIC)) AS dif_valor,
  
  -- Indicador de Consistência
  CASE 
    WHEN s.material IS NULL THEN 'Presente Apenas no Vetor'
    WHEN v.`código_item_vetor` IS NULL THEN 'Presente Apenas no SAP'
    WHEN ABS(SAFE_CAST(v.valor_total AS NUMERIC) - SAFE_CAST(s.valutilizlivre AS NUMERIC)) > 100 THEN 'Divergência Relevante'
    ELSE 'Consistente'
  END AS status_divergencia

FROM `cs-demo-2026.cs_frotas_data.relatorio_item_vetor` v
FULL OUTER JOIN `cs-demo-2026.cs_frotas_data.relatorio_estoque_sap_mb52` s
  ON v.`código_item_sap` = s.material;

## 5. Auditoria Inteligente de Anomalias com Gemini (BigQuery ML)

In [ ]:
%%bigquery --project cs-demo-2026
-- Criar modelo Gemini remoto no BigQuery ML
CREATE OR REPLACE MODEL `cs-demo-2026.cs_frotas_data.gemini_flash_model`
REMOTE WITH CONNECTION DEFAULT
OPTIONS(endpoint = 'gemini-1.5-flash');

In [ ]:
%%bigquery --project cs-demo-2026 df_resultado_ai
-- Gerar análises generativas sobre sobrepreço e incoerências
SELECT
  ml_generate_text_result AS analise_gemini,
  codigo_item_vetor,
  valor_total_vetor,
  valor_total_sap
FROM
  ML.GENERATE_TEXT(
    MODEL `cs-demo-2026.cs_frotas_data.gemini_flash_model`,
    (
      SELECT
        CONCAT(
          'Atue como um auditor de frota e suprimentos. Analise o item abaixo e explique a divergência:
',
          'Item Vetor: ', IFNULL(descricao_vetor, 'N/A'), ' | Valor Vetor: R$ ', CAST(IFNULL(valor_total_vetor, 0) AS STRING), '
',
          'Item SAP: ', IFNULL(descricao_sap, 'N/A'), ' | Valor SAP: R$ ', CAST(IFNULL(valor_total_sap, 0) AS STRING), '
',
          'Diferença de Valor: R$ ', CAST(IFNULL(dif_valor, 0) AS STRING)
        ) AS prompt,
        codigo_item_vetor,
        valor_total_vetor,
        valor_total_sap
      FROM `cs-demo-2026.cs_frotas_data.vw_cruzamento_vetor_sap`
      WHERE status_divergencia = 'Divergência Relevante'
      LIMIT 5
    ),
    STRUCT(0.2 AS temperature, 300 AS max_output_tokens)
  );

In [ ]:
# Exibir resultados gerados pelo Gemini
display(df_resultado_ai)